# Embico — модель оценки бластоцисты по фото · **v0.2**
**Что нового по сравнению с v0.1**
1. **5-fold кросс-валидация по пациенткам + ансамбль из 5 моделей** — стабильнее и точнее, обучение на всех данных.
2. **Два режима обучения:**
   - `strict` — из обучения **убраны пациентки, которые есть в тесте** → весь тест из 300 снимков «чистый», честная оценка;
   - `official` — официальное разбиение авторов → прямое сравнение со статьёй.
3. **Исправлена путаница EXP 3↔4** (убраны веса классов для EXP).
4. **Доверительные интервалы (bootstrap)** и вероятность того, что мы лучше Xception.
5. **Проверка защитного механизма:** точность на уверенных ответах vs на «нет рекомендации».

Режим: исследование / демо. Решение принимает эмбриолог. Запуск: Kaggle, GPU T4, Internet On, Run All (~1.5 ч на оба режима; для быстрого запуска оставьте в `MODES` один режим).

## 0. Настройки

In [ ]:
import os, re, glob, json, random, time, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, f1_score, accuracy_score, cohen_kappa_score, confusion_matrix
import matplotlib.pyplot as plt

CFG = dict(
    data_root   = '/kaggle/input',
    out_dir     = '/kaggle/working',
    MODES       = ['strict', 'official'],
    backbone    = 'efficientnet_b2',
    img_size    = 320,
    batch_size  = 32,
    n_folds     = 5,
    epochs      = 30,
    patience    = 7,
    lr          = 3e-4,
    weight_decay= 1e-4,
    seed        = 42,
    num_workers = 2,
    n_boot      = 1000,
    min_confidence = 0.55,
    model_version  = 'gardner-photo-v0.2.1',
)
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(CFG['out_dir'], exist_ok=True)
print('device:', DEVICE)

## 1. Файлы и метки (как в v0.1)

In [ ]:
def find_one(pattern):
    hits = glob.glob(os.path.join(CFG['data_root'], '**', pattern), recursive=True)
    assert hits, f'Не найден {pattern}'; return hits[0]
TRAIN_CSV, GOLD_CSV = find_one('Gardner_train_silver.csv'), find_one('Gardner_test_gold_onlyGardnerScores.csv')
CLIN_CSV, XCEP_CSV  = find_one('Clincial_annotations.csv'), find_one('prediction_xception.csv')
IMG_PATHS = {}
for ext in ('png', 'jpg', 'jpeg', 'bmp', 'tif'):
    for p in glob.glob(os.path.join(CFG['data_root'], '**', f'*.{ext}'), recursive=True):
        IMG_PATHS[os.path.basename(p)] = p

def read_semicolon(path):
    df = pd.read_csv(path, sep=';', encoding='latin-1', dtype=str)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]; df.columns = [c.strip() for c in df.columns]; return df
def pick(df, key): return [c for c in df.columns if c.upper().startswith(key)][0]
def to_int(v, valid):
    v = str(v).strip(); return int(v) if v in {str(x) for x in valid} else -1
def clean_labels(df):
    img = [c for c in df.columns if 'IMAGE' in c.upper()][0]
    out = pd.DataFrame({'image': df[img].str.strip()})
    out['exp'] = df[pick(df, 'EXP')].map(lambda v: to_int(v, range(5)))
    out['icm'] = df[pick(df, 'ICM')].map(lambda v: to_int(v, range(3)))
    out['te']  = df[pick(df, 'TE')].map(lambda v: to_int(v, range(3)))
    out.loc[out.exp.isin([0, 1]), ['icm', 'te']] = -1
    out['patient'] = out.image.str.split('_').str[0].str.lstrip('0')
    out['path'] = out.image.map(IMG_PATHS.get); return out

train_all = clean_labels(read_semicolon(TRAIN_CSV))
gold_raw = read_semicolon(GOLD_CSV); gold = clean_labels(gold_raw)
train_all = train_all[train_all.path.notna() & ~train_all.image.isin(gold.image)].reset_index(drop=True)
gold = gold[gold.path.notna()].reset_index(drop=True)
shared = set(train_all.patient) & set(gold.patient)
print(f'train {len(train_all)} снимков / {train_all.patient.nunique()} пациенток; test {len(gold)}; общих пациенток {len(shared)}')
print(f'в режиме strict из обучения уйдёт {train_all.patient.isin(shared).sum()} снимков')

## 1b. Проверка качества данных (препроцессинг)
- дубликаты строк и **противоречивые метки** (одна картинка — разные оценки);
- нечитаемые / битые картинки, размеры, «пустые» снимки;
- **дубликаты по содержимому** (одинаковые картинки под разными именами) между train и test — скрытая утечка.

In [ ]:
# 1) дубликаты имён: если оценки расходятся — убираем снимок целиком
dup = train_all[train_all.image.duplicated(keep=False)]
if len(dup):
    print('дубли в train:'); print(dup[['image', 'exp', 'icm', 'te']])
    conflict = dup.groupby('image')[['exp', 'icm', 'te']].nunique().max(axis=1) > 1
    train_all = train_all[~train_all.image.isin(conflict[conflict].index)].drop_duplicates('image').reset_index(drop=True)
    print(f'удалено противоречивых: {conflict.sum()}; train теперь {len(train_all)}')

# 2) читаемость, размер, яркость, перцептивный хэш
def img_info(p):
    try:
        im = Image.open(p); im.load(); g = np.asarray(im.convert('L').resize((64, 64)), dtype=np.float32)
        h = np.asarray(im.convert('L').resize((16, 16)), dtype=np.float32); bits = (h > h.mean()).ravel()
        return dict(ok=True, w=im.size[0], h=im.size[1], mean=g.mean(), std=g.std(), hash=np.packbits(bits).tobytes().hex())
    except Exception as e:
        return dict(ok=False, w=0, h=0, mean=np.nan, std=np.nan, hash=None)
info_tr = pd.DataFrame([img_info(p) for p in train_all.path]); info_te = pd.DataFrame([img_info(p) for p in gold.path])
print('битых: train', (~info_tr.ok).sum(), ' test', (~info_te.ok).sum())
print('размеры train:', info_tr.groupby(['w', 'h']).size().to_dict())
blank = info_tr['std'] < 5; print('почти пустых (std<5):', blank.sum())
train_all = train_all[info_tr.ok.values & ~blank.values].reset_index(drop=True); info_tr = info_tr[info_tr.ok.values & ~blank.values].reset_index(drop=True)

# 3) одинаковые картинки по содержимому
h_tr, h_te = set(info_tr.hash), set(info_te.hash)
print('одинаковых по содержимому внутри train:', info_tr.hash.duplicated().sum(), '| train∩test:', len(h_tr & h_te))
if h_tr & h_te:
    leak = info_tr.hash.isin(h_te).values; train_all = train_all[~leak].reset_index(drop=True)
    print(f'убрали из train {leak.sum()} снимков, совпадающих с тестом')
shared = set(train_all.patient) & set(gold.patient)
print(f'итог: train {len(train_all)}, test {len(gold)}, общих пациенток {len(shared)}')

## 2. Датасет, аугментации, модель

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225); S = CFG['img_size']
tf_train = T.Compose([T.Grayscale(3), T.RandomResizedCrop(S, scale=(0.8, 1.0), ratio=(0.95, 1.05)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(180),
    T.ColorJitter(brightness=0.2, contrast=0.2), T.ToTensor(), T.Normalize(MEAN, STD)])
tf_eval = T.Compose([T.Grayscale(3), T.Resize((S, S)), T.ToTensor(), T.Normalize(MEAN, STD)])

class BlastoDS(Dataset):
    def __init__(self, df, tf): self.df, self.tf = df.reset_index(drop=True), tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(r.path).convert('RGB')), torch.tensor([r.exp, r.icm, r.te])

class GardnerNet(nn.Module):
    def __init__(self, backbone, pretrained=True):
        super().__init__()
        self.body = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        d = self.body.num_features; self.drop = nn.Dropout(0.3)
        self.exp, self.icm, self.te = nn.Linear(d, 5), nn.Linear(d, 3), nn.Linear(d, 3)
    def forward(self, x):
        f = self.drop(self.body(x)); return self.exp(f), self.icm(f), self.te(f)

def soft_weights(series, n, power=0.5):
    c = series[series >= 0].value_counts().reindex(range(n), fill_value=0).values.astype(float)
    return torch.tensor((c.sum() / (n * np.maximum(c, 1))) ** power, dtype=torch.float32, device=DEVICE)

def make_loss(df):
    # v0.2: для EXP веса НЕ используем (в v0.1 они вызывали ложные «4»); для ICM/TE — мягкие веса
    crit = [nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=0.05),
            nn.CrossEntropyLoss(weight=soft_weights(df.icm, 3), ignore_index=-1, label_smoothing=0.05),
            nn.CrossEntropyLoss(weight=soft_weights(df.te, 3), ignore_index=-1, label_smoothing=0.05)]
    def loss_fn(outs, y):
        total = 0.
        for k, (o, c) in enumerate(zip(outs, crit)):
            if (y[:, k] >= 0).any(): total = total + c(o, y[:, k])
        return total
    return loss_fn

@torch.no_grad()
def predict_probs(models, loader, tta=True):
    for m in models: m.eval()
    P = [[], [], []]
    for x, _ in loader:
        x = x.to(DEVICE); views = [x, x.flip(-1), x.flip(-2), x.flip(-1).flip(-2)] if tta else [x]
        acc, n = None, 0
        for m in models:
            for v in views:
                with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                    o = [F.softmax(t.float(), 1) for t in m(v)]
                acc = o if acc is None else [a + b for a, b in zip(acc, o)]; n += 1
        for k in range(3): P[k].append((acc[k] / n).cpu())
    return [torch.cat(p).numpy() for p in P]

def head_f1(P, df):
    Y = df[['exp', 'icm', 'te']].values; s = {}
    for k, name in enumerate(['exp', 'icm', 'te']):
        m = Y[:, k] >= 0; s[name] = f1_score(Y[m, k], P[k][m].argmax(1), average='macro')
    s['mean'] = float(np.mean(list(s.values()))); return s

## 3. Обучение одного фолда

In [ ]:
def train_fold(tr_df, va_df, tag):
    seed_all(CFG['seed'])
    dl_tr = DataLoader(BlastoDS(tr_df, tf_train), CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'], drop_last=len(tr_df) > 2 * CFG['batch_size'])
    dl_va = DataLoader(BlastoDS(va_df, tf_eval), CFG['batch_size'], num_workers=CFG['num_workers'])
    model = GardnerNet(CFG['backbone']).to(DEVICE); loss_fn = make_loss(tr_df)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['lr'], total_steps=CFG['epochs'] * len(dl_tr), pct_start=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')
    best, bad, best_state = -1, 0, None
    for ep in range(CFG['epochs']):
        model.train(); t0 = time.time()
        for x, y in dl_tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                loss = loss_fn(model(x), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        s = head_f1(predict_probs([model], dl_va, tta=False), va_df)
        flag = ''
        if s['mean'] > best:
            best, bad, best_state, flag = s['mean'], 0, copy.deepcopy(model.state_dict()), ' *'
        else:
            bad += 1
        print(f"  [{tag}] ep {ep+1:02d} val F1 exp {s['exp']:.3f} icm {s['icm']:.3f} te {s['te']:.3f} mean {s['mean']:.3f} {time.time()-t0:.0f}s{flag}")
        if bad >= CFG['patience']: break
    model.load_state_dict(best_state); model.eval()
    oof = predict_probs([model], dl_va, tta=True)
    return model, oof, best

## 4. Оценка (логика скрипта авторов) + bootstrap
ICM/TE не учитываются, если EXP ∈ {0,1} у модели или у экспертов; «не оценивается» — отдельный класс.

In [ ]:
def gt_int(v):
    v = str(v).strip()
    if v in ('', 'nan', 'NA'): return -1
    if v == 'ND': return 3
    return int(v) if v.lstrip('-').isdigit() else -1
GOLD = {r.iloc[0].strip(): [gt_int(x) for x in r.iloc[1:4]] for _, r in gold_raw.iterrows()}

def records(pred):
    """pred: DataFrame [image, exp, icm, te] -> по картинкам: e, eg, i, ig, t, tg, use_icm"""
    rows = []
    for _, r in pred.iterrows():
        fn = str(r.iloc[0]).strip()
        if fn not in GOLD: continue
        eg, ig, tg = GOLD[fn]
        e, i, t = [int(x) if pd.notna(x) else -1 for x in r.iloc[1:4]]
        i = i if i in (0, 1, 2) else 3; t = t if t in (0, 1, 2) else 3
        ig = 3 if ig == -1 else ig; tg = 3 if tg == -1 else tg
        e = e if e in range(5) else 5; eg = eg if eg in range(5) else 5
        rows.append((fn, e, eg, i, ig, t, tg, not (e in (0, 1) or eg in (0, 1))))
    return pd.DataFrame(rows, columns=['image', 'e', 'eg', 'i', 'ig', 't', 'tg', 'use']).set_index('image')

def metrics(R):
    out = {}
    for name, p, g, mask in [('EXP', 'e', 'eg', None), ('ICM', 'i', 'ig', 'use'), ('TE', 't', 'tg', 'use')]:
        d = R if mask is None else R[R[mask]]
        if len(d) == 0: out.update({f'{name}_acc': np.nan, f'{name}_F1': np.nan, f'{name}_QWK': np.nan}); continue
        out[f'{name}_acc'] = accuracy_score(d[g], d[p])
        out[f'{name}_F1'] = f1_score(d[g], d[p], average='macro')
        out[f'{name}_QWK'] = cohen_kappa_score(d[g], d[p], weights='quadratic')
    return out

def bootstrap_compare(R_ours, R_ref, n_boot, seed=0):
    idx = R_ours.index.intersection(R_ref.index).values; rng = np.random.default_rng(seed)
    base_o, base_r = metrics(R_ours.loc[idx]), metrics(R_ref.loc[idx])
    bo, br = [], []
    for _ in range(n_boot):
        s = rng.choice(idx, len(idx), replace=True)
        bo.append(metrics(R_ours.loc[s])); br.append(metrics(R_ref.loc[s]))
    bo, br = pd.DataFrame(bo), pd.DataFrame(br)
    tab = pd.DataFrame({'Embico': pd.Series(base_o), 'Embico 95% ДИ': [f'{bo[c].quantile(.025):.3f}–{bo[c].quantile(.975):.3f}' for c in bo],
                        'Xception': pd.Series(base_r), 'Xception 95% ДИ': [f'{br[c].quantile(.025):.3f}–{br[c].quantile(.975):.3f}' for c in br],
                        'P(Embico > Xception)': (bo > br).mean().round(3)})
    return tab.round(3)

## 5. Запуск: 5 фолдов → ансамбль → тест (для каждого режима)

In [ ]:
xcep = pd.read_csv(XCEP_CSV, header=None); R_x = records(xcep)
dl_test = DataLoader(BlastoDS(gold, tf_eval), CFG['batch_size'], num_workers=CFG['num_workers'])
RESULTS, ENSEMBLES, TEST_PROBS = {}, {}, {}

for mode in CFG['MODES']:
    print(f'\n================ режим: {mode} ================')
    df = train_all[~train_all.patient.isin(shared)].reset_index(drop=True) if mode == 'strict' else train_all.copy()
    print(f'обучающих снимков: {len(df)}, пациенток: {df.patient.nunique()}')
    sgkf = StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
    oof = [np.zeros((len(df), 5)), np.zeros((len(df), 3)), np.zeros((len(df), 3))]
    models = []
    for f, (tr, va) in enumerate(sgkf.split(df, df.exp, groups=df.patient)):
        assert not set(df.patient.iloc[tr]) & set(df.patient.iloc[va])
        m, p, best = train_fold(df.iloc[tr], df.iloc[va], f'{mode} fold {f+1}')
        for k in range(3): oof[k][va] = p[k]
        models.append(m); torch.save(m.state_dict(), os.path.join(CFG['out_dir'], f'gardner_{mode}_fold{f+1}.pt'))
        print(f'  fold {f+1}: best val mean F1 = {best:.3f}')
    s = head_f1(oof, df)
    print(f"OOF (вся обучающая выборка, 5 фолдов): F1 exp {s['exp']:.3f} icm {s['icm']:.3f} te {s['te']:.3f} mean {s['mean']:.3f}")
    P = predict_probs(models, dl_test, tta=True); TEST_PROBS[mode] = P; ENSEMBLES[mode] = models
    pred = pd.DataFrame({'image': gold.image, 'exp': P[0].argmax(1), 'icm': P[1].argmax(1), 'te': P[2].argmax(1)})
    pred.to_csv(os.path.join(CFG['out_dir'], f'prediction_embico_{mode}.csv'), header=False, index=False)
    R = records(pred); RESULTS[mode] = R
    print('\nТест 300 снимков (консенсус экспертов) — ансамбль из 5 моделей vs Xception:')
    tab = bootstrap_compare(R, R_x, CFG['n_boot']); display(tab)
    tab.to_csv(os.path.join(CFG['out_dir'], f'test_results_{mode}.csv'))

## 6. Сводка и матрицы ошибок
**Как читать:** в режиме `strict` модель НЕ видела ни одной тестовой пациентки, а Xception видела 200 из 249 — сравнение консервативное (в пользу Xception). Если `strict` ≈ Xception — это сильный результат.

In [ ]:
summary = pd.DataFrame({m: metrics(R) for m, R in RESULTS.items()} | {'Xception': metrics(R_x)}).round(3)
display(summary); summary.to_csv(os.path.join(CFG['out_dir'], 'summary_v02.csv'))

for mode, R in RESULTS.items():
    fig, ax = plt.subplots(1, 3, figsize=(15, 4)); fig.suptitle(f'{mode}: эксперт (строки) vs модель (столбцы)')
    for a, (name, p, g, mask) in zip(ax, [('EXP', 'e', 'eg', None), ('ICM', 'i', 'ig', 'use'), ('TE', 't', 'tg', 'use')]):
        d = R if mask is None else R[R[mask]]
        labels = sorted(set(d[g]) | set(d[p])); cm = confusion_matrix(d[g], d[p], labels=labels)
        a.imshow(cm, cmap='Blues'); a.set_title(name); a.set_xticks(range(len(labels)), labels); a.set_yticks(range(len(labels)), labels)
        for (r, c), v in np.ndenumerate(cm): a.text(c, r, v, ha='center', va='center')
    plt.tight_layout(); plt.show()

## 7. Работает ли защита «нет рекомендации»?
Хорошая защита = на уверенных ответах точность заметно выше, чем на неуверенных.

In [ ]:
for mode, P in TEST_PROBS.items():
    R = RESULTS[mode].loc[gold.image]
    conf_e = P[0].max(1); conf_i = np.where(R.use, P[1].max(1), 1); conf_t = np.where(R.use, P[2].max(1), 1)
    conf = np.minimum.reduce([conf_e, conf_i, conf_t]); ok = conf >= CFG['min_confidence']
    corr = (R.e.values == R.eg.values) & (~R.use.values | ((R.i.values == R.ig.values) & (R.t.values == R.tg.values)))
    print(f'[{mode}] уверенных ответов: {ok.mean():.0%} | полностью верная оценка: уверенные {corr[ok].mean():.0%}, '
          f'«нет рекомендации» {corr[~ok].mean() if (~ok).any() else float("nan"):.0%}')

## 8. Функция для платформы (ансамбль режима `strict`)

In [ ]:
ICM_TE = {0: 'A', 1: 'B', 2: 'C'}
SERVE_MODE = 'strict' if 'strict' in ENSEMBLES else CFG['MODES'][0]

@torch.no_grad()
def predict_embryo(img_path, models=None):
    models = models or ENSEMBLES[SERVE_MODE]
    img = Image.open(img_path).convert('RGB'); warnings = []
    if min(img.size) < 200: warnings.append('низкое разрешение снимка')
    x = tf_eval(img).unsqueeze(0).to(DEVICE)
    views = [x, x.flip(-1), x.flip(-2), x.flip(-1).flip(-2)]
    outs = [[F.softmax(o.float(), 1) for o in m.eval()(v)] for m in models for v in views]
    pe, pi, pt = [torch.stack([o[k] for o in outs]).mean(0)[0].cpu().numpy() for k in range(3)]
    exp = int(pe.argmax()); conf = {'exp': float(pe.max())}
    out = {'model_version': CFG['model_version'] + f'-{SERVE_MODE}-ens{len(models)}', 'task': 'gardner_grading_photo',
           'expansion_class': exp, 'probabilities': {'exp': pe.round(3).tolist()}}
    if exp >= 2:
        icm, te = int(pi.argmax()), int(pt.argmax()); conf.update(icm=float(pi.max()), te=float(pt.max()))
        out.update(icm=ICM_TE[icm], te=ICM_TE[te], gardner=f'EXP{exp}-{ICM_TE[icm]}{ICM_TE[te]}')
        out['probabilities'].update(icm=pi.round(3).tolist(), te=pt.round(3).tolist())
    else:
        out.update(icm=None, te=None, gardner=f'EXP{exp} (ICM/TE не оцениваются на этой стадии)')
    out['confidence'] = round(min(conf.values()), 3)
    low = out['confidence'] < CFG['min_confidence']
    if low: warnings.append('низкая уверенность модели — нужна оценка эмбриолога')
    out.update(status='no_recommendation' if (low or warnings) else 'ok', warnings=warnings,
               disclaimer='Только для исследования и проверки эмбриологом. Не является клиническим решением.')
    return out

print(json.dumps(predict_embryo(gold.path[0]), ensure_ascii=False, indent=2))

## 9. Итог
Файлы в `/kaggle/working`: веса 5 фолдов для каждого режима, `prediction_embico_<mode>.csv` (формат авторов), `test_results_<mode>.csv` (с ДИ), `summary_v02.csv`.
**Следующий шаг:** сравнить с отдельными экспертами (`Gardner_test_gold.xlsx`) — «модель vs эмбриолог».